## Purpose

M. B. Lagutin's *Visual Mathematical Statistics* is full of small experiments where a simple probabilistic model explains a surprisingly practical effect. This notebook turns three of those observations into reproducible numerical sketches: Dorfman's pooled testing for syphilis screening, the paradoxical experiment called "Failures", and a warning about Euler's method near a singular differential equation.

Source note: the book is available as a public catalogue page at [LitRes](https://www.litres.ru/book/m-b-lagutin-6593695/naglyadnaya-matematicheskaya-statistika-6713832/) and is also listed on [Google Books](https://books.google.com/books/about/%D0%9D%D0%B0%D0%B3%D0%BB%D1%8F%D0%B4%D0%BD%D0%B0%D1%8F_%D0%BC%D0%B0%D1%82%D0%B5%D0%BC%D0%B0%D1%82%D0%B8%D1%87%D0%B5.html?id=t72WswEACAAJ).


In [14]:
import math

import numpy as np
import plotly.graph_objects as go
from IPython.display import HTML

RNG = np.random.default_rng(42)

## 1. Dorfman testing and the syphilis screening problem

Lagutin describes a wartime medical-screening problem. The Wassermann reaction can detect antibodies associated with syphilis. Instead of testing every person separately, Robert Dorfman's idea was to pool blood samples from `k` people and test the mixture once. If the mixture is negative, one test clears the whole group. If it is positive, the group is opened and all `k` people are tested individually, so that group costs `k+1` tests.

Here is the derivation of the expected-test formula. For one group of `k` people, let `X_j` be the number of tests used by this group.

The probability that all `k` people in the group are healthy is

$$
P(\text{negative pool})=(1-p)^k,
$$

because every person is healthy with probability `1-p` and the model assumes independence. In this case the group needs only the pooled test, so `X_j=1`.

The complementary event is that at least one person is antibody-positive:

$$
P(\text{positive pool})=1-(1-p)^k.
$$

Then the pooled test is followed by `k` individual tests, so `X_j=k+1`. Therefore the expected number of tests for one group is

$$
\begin{aligned}
E[X_j]
&=1\cdot(1-p)^k+(k+1)\cdot\bigl(1-(1-p)^k\bigr)\\
&=(1-p)^k+k+1-(k+1)(1-p)^k\\
&=k+1-k(1-p)^k.
\end{aligned}
$$

If there are `n` people and `n` is divisible by `k`, there are `n/k` groups. The total number of tests is `Z=X_1+...+X_{n/k}`, so by linearity of expectation

$$
\begin{aligned}
E[Z]
&=\frac{n}{k}E[X_j]\\
&=\frac{n}{k}\left(k+1-k(1-p)^k\right)\\
&=n\left(1+\frac{1}{k}-(1-p)^k\right).
\end{aligned}
$$

Dividing by `n` gives the expected number of tests per person:

$$
H(k)=\frac{E[Z]}{n}=1+\frac{1}{k}-(1-p)^k.
$$

Equivalently, this is `1/k` pooled tests per person plus `1-(1-p)^k` expected follow-up tests per person. For rare disease, `(1-p)^k` is close to `1-pk`, so `H(k)` behaves like `1/k+pk`. This approximation is minimized near `k=1/sqrt(p)`. With `p=0.01`, that gives `k` near 10 and about `0.2` tests per person, which is the fivefold saving emphasized in the book. The method fails when `k` is too large: then almost every pool is positive, so the first pooled test becomes mostly overhead.


In [15]:
p_values = np.array([0.001, 0.003, 0.01, 0.03, 0.05], dtype=float)
k_values = np.arange(2, 61)

fig_group = go.Figure()
for p in p_values:
    expected_tests = 1 + 1 / k_values - (1 - p) ** k_values
    best_idx = int(np.argmin(expected_tests))
    fig_group.add_trace(
        go.Scatter(
            x=k_values,
            y=expected_tests,
            mode="lines",
            name=f"p={p:g}",
            hovertemplate="group size=%{x}<br>tests/person=%{y:.3f}<extra></extra>",
        )
    )
    fig_group.add_trace(
        go.Scatter(
            x=[int(k_values[best_idx])],
            y=[float(expected_tests[best_idx])],
            mode="markers",
            marker={"size": 9},
            showlegend=False,
            hovertemplate=(
                f"best integer k={int(k_values[best_idx])}<br>"
                "tests/person=%{y:.3f}<extra></extra>"
            ),
        )
    )

fig_group.add_trace(
    go.Scatter(
        x=[int(k_values[0]), int(k_values[-1])],
        y=[1, 1],
        mode="lines",
        line={"dash": "dash", "color": "black"},
        name="individual testing",
        hovertemplate="individual baseline: 1 test/person<extra></extra>",
    )
)
fig_group.update_layout(
    title="Dorfman pooled testing: expected tests per person",
    xaxis_title="group size k",
    yaxis_title="expected tests per person",
    template="plotly_white",
    legend_title="disease probability",
)
fig_group.update_yaxes(range=[0, 1.08])
HTML(fig_group.to_html(include_plotlyjs="cdn", full_html=False))

## 2. The "Failures" experiment and finite precision

The second example asks: if `X0` is the size of my failure and `X1, X2, ...` are independent failures of other people from the same continuous distribution, how many people must I ask until I meet someone whose failure is at least as large as mine?

Let

$$
N=\min\{n\ge 1: X_n\ge X_0\}.
$$

In the continuous model there are no ties. Among `X0, X1, ..., Xn`, every observation is equally likely to be the maximum, so

$$
P(N>n)=P(X_0=\max\{X_0,\ldots,X_n\})=
rac{1}{n+1}.
$$

Therefore `E[N]` is the harmonic series and diverges. A short simulation with ordinary-looking random numbers does not feel infinite because the physical experiment is not continuous. If the numbers are rounded to `d` decimals, there are only `m=10^d` distinct values. Conditional on `X0=j/m`, success has probability `(m-j)/m`, and averaging the geometric means gives

$$
E[N_d]=\sum_{r=1}^{m}
\frac{1}{r}=H_m \approx d\ln(10)+\gamma.
$$

This is Lagutin's key diagnostic point: when practice compares `X_i` only to a fixed precision, the right theory is the finite-precision theory. With two decimals, the target is about `H_100=5.19`, not infinity.


In [16]:
euler_gamma = 0.5772156649015329
precisions = np.arange(1, 6)
m_values = 10**precisions

harmonic = np.array([np.sum(1.0 / np.arange(1, m + 1)) for m in m_values])
approximation = precisions * np.log(10) + euler_gamma

trials = 200_000
simulated_means = []
for m in m_values:
    x0 = RNG.integers(0, m, size=trials)
    success_probability = (m - x0) / m
    waits = RNG.geometric(success_probability)
    simulated_means.append(float(np.mean(waits)))
simulated_means = np.array(simulated_means)

fig_failures = go.Figure()
fig_failures.add_trace(
    go.Scatter(
        x=precisions,
        y=harmonic,
        mode="lines+markers",
        name="finite-precision theory H_m",
        hovertemplate="digits=%{x}<br>mean=%{y:.3f}<extra></extra>",
    )
)
fig_failures.add_trace(
    go.Scatter(
        x=precisions,
        y=approximation,
        mode="lines+markers",
        line={"dash": "dash"},
        name="log approximation",
        hovertemplate="digits=%{x}<br>d ln(10)+gamma=%{y:.3f}<extra></extra>",
    )
)
fig_failures.add_trace(
    go.Scatter(
        x=precisions,
        y=simulated_means,
        mode="markers",
        marker={"size": 11, "symbol": "diamond"},
        name="simulation",
        hovertemplate="digits=%{x}<br>simulated mean=%{y:.3f}<extra></extra>",
    )
)
fig_failures.update_layout(
    title="The 'Failures' experiment depends on numeric precision",
    xaxis_title="decimal digits retained",
    yaxis_title="mean waiting count",
    template="plotly_white",
    hovermode="x unified",
)
HTML(fig_failures.to_html(include_plotlyjs="cdn", full_html=False))

## 3. Euler's method and a differential-equation trap

Lagutin's numerical example is the initial-value problem

$$
y'=-
\frac{x}{y}, \qquad y(-1)=0.21.
$$

Separating variables gives `y^2+x^2=1.0441`, so the positive theoretical branch is

$$
y=\sqrt{1.0441-x^2}.
$$

The catch is that the right side `-x/y` is singular at `y=0`. Euler's method assumes that the differential equation is well behaved along the path being approximated. Near `y=0`, the slope is huge. A coarse step can cross the singular line and land on the wrong side, after which the recurrence is no longer following the theoretical branch.


In [17]:
c = 1.0 + 0.21**2
x_start = -1.0
y_start = 0.21
x_boundary = math.sqrt(c)


def exact_upper(xs: np.ndarray) -> np.ndarray:
    values = c - xs**2
    return np.where(values >= 0, np.sqrt(np.clip(values, 0, None)), np.nan)


def euler_path(step: float, x_end: float = 1.55) -> tuple[np.ndarray, np.ndarray]:
    xs = [x_start]
    ys = [y_start]
    x = x_start
    y = y_start
    while x < x_end - 1e-12:
        y = y + step * (-x / y)
        x = x + step
        xs.append(x)
        ys.append(y)
        if len(xs) > 20_000:
            break
    return np.array(xs), np.array(ys)


def transformed_z_path(step: float) -> tuple[np.ndarray, np.ndarray]:
    xs = [x_start]
    ys = [y_start]
    x = x_start
    z = y_start**2
    while x < x_boundary - 1e-12:
        next_x = min(x + step, x_boundary)
        next_z = z - (next_x**2 - x**2)
        if next_z <= 0:
            xs.append(x_boundary)
            ys.append(0.0)
            break
        xs.append(next_x)
        ys.append(math.sqrt(next_z))
        x = next_x
        z = next_z
    return np.array(xs), np.array(ys)


fig_euler = go.Figure()
xs_exact = np.linspace(x_start, x_boundary, 500)
fig_euler.add_trace(
    go.Scatter(
        x=xs_exact,
        y=exact_upper(xs_exact),
        mode="lines",
        line={"width": 4, "color": "black"},
        name="exact upper branch",
        hovertemplate="x=%{x:.3f}<br>y=%{y:.3f}<extra></extra>",
    )
)
for step in [0.10, 0.05, 0.02]:
    xs, ys = euler_path(step)
    fig_euler.add_trace(
        go.Scatter(
            x=xs,
            y=ys,
            mode="lines+markers",
            name=f"Euler h={step:.2f}",
            hovertemplate="x=%{x:.3f}<br>y=%{y:.3f}<extra></extra>",
        )
    )
xs_z, ys_z = transformed_z_path(0.10)
fig_euler.add_trace(
    go.Scatter(
        x=xs_z,
        y=ys_z,
        mode="lines+markers",
        line={"dash": "dot", "width": 3},
        marker={"symbol": "square"},
        name="z=y^2 step with boundary stop",
        hovertemplate="x=%{x:.3f}<br>y=%{y:.3f}<extra></extra>",
    )
)
fig_euler.add_trace(
    go.Scatter(
        x=[x_start, 1.55],
        y=[0, 0],
        mode="lines",
        line={"dash": "dash", "color": "gray"},
        name="singular line y=0",
        hoverinfo="skip",
    )
)
fig_euler.update_layout(
    title="Euler's method can jump across y=0 for y'=-x/y",
    xaxis_title="x",
    yaxis_title="y",
    template="plotly_white",
)
fig_euler.update_yaxes(range=[-0.8, 1.4])
HTML(fig_euler.to_html(include_plotlyjs="cdn", full_html=False))

The practical repair is to make the numerical problem match the assumptions behind convergence. On any interval that stays away from `y=0`, smaller Euler steps converge to the theoretical branch. Near the boundary, the solver should use event detection and stop when `y` reaches zero, or change variables to `z=y^2`. Then

$$
z'=2yy'=-2x,
$$

which is smooth through the boundary. The dotted path uses this transformed variable and stops at the theoretical endpoint instead of stepping into the invalid half-plane. The general lesson is the same as in the finite-precision experiment: simulation and theory agree when the computational model is the same model the theorem is talking about.
